# GAS4 — Member 5: common computational cost

Scope: independent computational measurements of the team's four already-selected pipelines on 22,191 development observations (2011–2013). The shared benchmarking implementation is reused, not claimed as original Member 5 implementation. No final held-out evaluation or hyperparameter retuning is performed here.

Recorded local run: 12 November 2026; Windows 11 (reported by Python as Windows-10-10.0.22631-SP0); 8 logical CPU cores; 7.65 GiB RAM; Python 3.10.6; scikit-learn 1.7.2; numerical thread limit of 1.

Result provenance: results/member5_local_py310/ contains Member 5's locally measured costs and environment. results/development/comparison.csv contains team development-validation scores from another runtime. results/costs/ contains earlier Codex integration reference costs, not Member 5's measurements.

In [2]:
import sys
import sklearn
import pandas
import numpy

print("Python:", sys.version)
print("Environment:", sys.executable)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Python: 3.10.6 (tags/v3.10.6:9c7b4bd, Aug  1 2022, 21:53:49) [MSC v.1932 64 bit (AMD64)]
Environment: e:\AI\ENG5338_CW2\.venv\Scripts\python.exe
Scikit-learn: 1.7.2
Pandas: 2.3.3
NumPy: 2.2.6


In [3]:
from pathlib import Path
import sys
# Support the project root, its experiments folder, or the parent VS Code workspace.
locations = [Path.cwd(), Path.cwd() / "ENG5338_CW2", Path.cwd().parent]
PROJECT = next((p for p in locations if (p / "src/cw2_project.py").is_file()), None)
assert PROJECT is not None, "Extract/download the whole repository and run from its root or experiments folder."
sys.path.insert(0, str(PROJECT / "src"))
from cw2_project import *
from IPython.display import display
THREAD_CONTROLLER = threadpool_limits(limits=1)
np.random.seed(SEED)


In [4]:

# Verify that the shared project is accessible

print("Project folder:", PROJECT)
print("Dataset exists:", (
    PROJECT / "data/gas-turbine-emissions.csv"
).exists())

print("Available models:", MODELS)
print("Shared protocol:", PROTOCOL_ID)

# Load development data only (2011-2013)
X, y, folds = load_development(
    PROJECT / "data/gas-turbine-emissions.csv"
)

print("Development rows:", len(X))
print("Input features:", X.shape[1])
print("Cross-validation folds:", len(folds))


Project folder: e:\AI\ENG5338_CW2
Dataset exists: True
Available models: ['linear_reference', 'random_forest', 'extra_trees', 'hist_gradient_boosting']
Shared protocol: 454f332babc7
Development rows: 22191
Input features: 9
Cross-validation folds: 5


In [5]:

from sklearn.base import clone
import numpy as np

# Check each model on a small portion of development data.
# This is only a compatibility test, not a benchmark.

X_small = X.iloc[:500]
y_small = y.iloc[:500]

for name in MODELS:
    print(f"\nChecking {name}...")

    pipeline = clone(pipeline_for(name))

    # Train on a small sample
    pipeline.fit(X_small, y_small)

    # Predict one observation from development data
    prediction = pipeline.predict(X.iloc[500:501])

    assert len(prediction) == 1
    assert np.isfinite(prediction).all()

    print("Model fit: successful")
    print("Prediction: successful")

print("\nAll compatibility checks passed!")



Checking linear_reference...
Model fit: successful
Prediction: successful

Checking random_forest...
Model fit: successful
Prediction: successful

Checking extra_trees...
Model fit: successful
Prediction: successful

Checking hist_gradient_boosting...
Model fit: successful
Prediction: successful

All compatibility checks passed!


## Measurement method and safeguards

The project function benchmark_all measures each unchanged selected pipeline on the same development data and runtime, with one warm-up and three timed full-data fits; a separately fitted pipeline for predictions; three batch-prediction warm-ups followed by ten 1,000-row batch timings; and five single-row warm-ups followed by 100 single-row timings. It reports medians, single-row p95, serialized pipeline size and an approximate RSS memory increment from a separate fit. Numerical library threads are limited to one.

Limitations: times are machine- and workload-dependent; Python and package versions also matter. RSS polling at about 5 ms intervals can miss peaks, and a reported 0 MiB increase does not mean zero memory consumption. Hyperparameter-search time is not included. The final held-out 2014–2015 data is not accessed in this notebook.

In [9]:
# Display for context; the local benchmark and its saved provenance are below.
reference_path = PROJECT / "results/costs/comparison.csv"
if reference_path.exists():
    print("Reference timings from earlier Codex integration (not Member 5 measurements):")
    display(pd.read_csv(reference_path)[[
        "model", "fit_median_s", "single_median_ms", "batch_1000_median_ms"
    ]].round(4))
else:
    print("Earlier integration-reference costs not found. Continuing with local measurements.")

Reference timings from earlier Codex integration (not Member 5 measurements):


,model,fit_median_s,single_median_ms,batch_1000_median_ms
0,linear_reference,0.0163,1.0241,1.1032
1,random_forest,23.8227,38.1850,67.2419
2,extra_trees,5.4586,26.6633,56.4639
3,hist_gradient_boosting,0.3345,2.5486,8.3841


In [12]:
# Member 5: local CPU cost measurements — load saved results on rerun.
# First execution ran the shared benchmark and produced the measured outputs
# preserved in this notebook. Subsequent executions do not retrain.
from pathlib import Path
from IPython.display import display
import pandas as pd

output_dir = PROJECT / "results" / "member5_local_py310"
csv_path = output_dir / "comparison.csv"
runtime_path = output_dir / "runtime.json"

if csv_path.is_file() and runtime_path.is_file():
    cost_results = pd.read_csv(csv_path)
    print("Loaded saved Windows measurements; benchmark was NOT rerun.")
elif output_dir.exists():
    raise RuntimeError(
        "An incomplete results folder exists. Inspect it before proceeding; "
        "do not overwrite previous experiment evidence."
    )
else:
    # Only executed when no measurements have been saved yet.
    import platform
    if platform.system() != "Windows":
        raise RuntimeError(
            "This cell stores Windows-only local results. For Colab, run "
            "benchmark_all() into results/member5_colab_verification instead."
        )
    cost_results = benchmark_all(
        output_dir,
        measured_by="Sauhandikaa - Windows Python 3.10.6"
    )
    print("New Windows benchmark completed and saved.")

columns = [
    "model", "fit_median_s", "single_median_ms",
    "batch_1000_median_ms", "serialized_pipeline_kib"
]
display(cost_results[columns].round(4))
print("Measurements directory:", output_dir)

Loaded saved Windows measurements; benchmark was NOT rerun.


,model,fit_median_s,single_median_ms,batch_1000_median_ms,serialized_pipeline_kib
0,linear_reference,0.0205,0.6952,0.8243,1.5957
1,random_forest,24.4929,27.1806,69.8337,24068.3047
2,extra_trees,5.2187,19.0255,56.0800,44718.1787
3,hist_gradient_boosting,0.5542,2.8636,19.1155,364.9746


Measurements directory: e:\AI\ENG5338_CW2\results\member5_local_py310


In [13]:
import json
import pandas as pd

# Load my saved benchmarking records
results_folder = output_dir

runtime = json.loads(
    (results_folder / "runtime.json").read_text()
)

costs = pd.read_csv(
    results_folder / "comparison.csv"
)

# Verify the benchmark results
assert len(costs) == 4
assert costs["protocol_id"].nunique() == 1
assert costs["protocol_id"].iloc[0] == PROTOCOL_ID
assert not costs["final_test_run"].any()

print("BENCHMARK ENVIRONMENT")
print("Operating system:", runtime["platform"])
print("Processor:", runtime["cpu"])
print("Logical CPU cores:", runtime["logical_cpus"])
print("RAM (GiB):", round(runtime["ram_gib"], 2))
print("Python:", runtime["python"].split()[0])
print("Scikit-learn:",
      runtime["versions"]["scikit-learn"])
print("Numerical threads:", runtime["numerical_threads"])

print("\nLATENCY CONSTRAINT CHECK")
latency_check = costs[
    ["model", "single_median_ms"]
].copy()

latency_check["Meets 10 ms target"] = (
    latency_check["single_median_ms"] <= 10
)

display(latency_check.round(4))

print("\nVerification successful.")
print("No held-out test was performed.")

BENCHMARK ENVIRONMENT
Operating system: Windows-10-10.0.22631-SP0
Processor: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel
Logical CPU cores: 8
RAM (GiB): 7.65
Python: 3.10.6
Scikit-learn: 1.7.2
Numerical threads: 1

LATENCY CONSTRAINT CHECK


,model,single_median_ms,Meets 10 ms target
0,linear_reference,0.6952,True
1,random_forest,27.1806,False
2,extra_trees,19.0255,False
3,hist_gradient_boosting,2.8636,True



Verification successful.
No held-out test was performed.


In [14]:
# Member 5: Combine development accuracy with my measured costs

import pandas as pd
from IPython.display import display

results_dir = output_dir

# These are existing team development-validation results
performance = pd.read_csv(
    PROJECT / "results/development/comparison.csv"
)

# These are my independently executed Windows benchmarks
costs = pd.read_csv(results_dir / "comparison.csv")

# Check protocol consistency and test isolation
assert set(performance["model"]) == set(costs["model"]) == set(MODELS)
assert performance["protocol_id"].eq(PROTOCOL_ID).all()
assert costs["protocol_id"].eq(PROTOCOL_ID).all()
assert not performance["final_test_run"].any()
assert not costs["final_test_run"].any()

comparison = performance[
    ["model", "train_mae", "validation_mae",
     "validation_coverage", "mae_gap"]
].merge(
    costs[
        ["model", "fit_median_s", "single_median_ms",
         "batch_1000_median_ms", "serialized_pipeline_kib"]
    ],
    on="model",
    validate="one_to_one"
)

comparison["coverage_percent"] = (
    comparison["validation_coverage"] * 100
)

comparison["coverage_pass"] = (
    comparison["validation_coverage"] >= 0.90
)

comparison["latency_pass"] = (
    comparison["single_median_ms"] <= 10
)

comparison["both_pass"] = (
    comparison["coverage_pass"] & comparison["latency_pass"]
)

comparison = comparison.sort_values("validation_mae")

display(comparison[
    ["model", "validation_mae", "coverage_percent",
     "fit_median_s", "single_median_ms",
     "coverage_pass", "latency_pass", "both_pass"]
].round(4))

comparison.to_csv(
    results_dir / "member5_comparison.csv",
    index=False
)

print("Member 5 comparison saved successfully.")
print("No final test data used.")

,model,validation_mae,coverage_percent,fit_median_s,single_median_ms,coverage_pass,latency_pass,both_pass
2,extra_trees,0.7226,81.7631,5.2187,19.0255,False,False,False
1,random_forest,0.7372,81.1790,24.4929,27.1806,False,False,False
3,hist_gradient_boosting,0.7655,79.1509,0.5542,2.8636,False,True,False
0,linear_reference,0.9073,71.8010,0.0205,0.6952,False,True,False


Member 5 comparison saved successfully.
No final test data used.


## Member 5 contribution and interpretation

### Measurement context

On 12 November 2026, I, Sauhandikaa Senthil Arumugam checked that all four selected pipelines could fit and predict with my local Python environment, then ran the project's existing benchmark_all() function to measure computational costs for the 22,191-row 2011–2013 development dataset. The same laptop and one numerical thread were used throughout: Windows 11 / Windows build 22631, eight logical CPU cores, 7.65 GiB RAM, Python 3.10.6, scikit-learn 1.7.2. The original benchmark samples and environment metadata are saved under results/member5_local_py310/ in my local repository.

### Computational findings

| Pipeline | Median training time (s) | Median single-row prediction (ms) | 1,000-row batch (ms) | Serialized size (KiB) |
|---|---:|---:|---:|---:|
| Linear Regression | 0.0205 | 0.6952 | 0.8243 | 1.5957 |
| Random Forest | 24.4929 | 27.1806 | 69.8337 | 24,068.3047 |
| Extra Trees | 5.2187 | 19.0255 | 56.0800 | 44,718.1787 |
| Histogram Gradient Boosting (HGBR) | 0.5542 | 2.8636 | 19.1155 | 364.9746 |

Linear Regression is quickest and smallest but has the worst recorded development-validation MAE (0.9073 mg/m³). Extra Trees achieves the best development-validation MAE (0.7226 mg/m³), but its median single-row latency is 19.03 ms. Random Forest is slower than Extra Trees in both fitting and single-row inference while having slightly worse validation MAE (0.7372 mg/m³). Among the nonlinear candidates, HGBR has the lowest measured fitting and inference costs (0.5542 s and 2.86 ms), at the cost of higher validation MAE (0.7655 mg/m³) than the other nonlinear candidates.

### Accuracy versus proposed constraints
Validation coverage within ±1 mg/m³ is 81.76% for Extra Trees, 81.18% for Random Forest, 79.15% for HGBR, and 71.80% for Linear Regression. On my laptop, only HGBR and Linear Regression meet the team's provisional 10 ms median single-row latency target. None achieves the provisional 90% development-validation coverage target. Consequently, no pipeline satisfies both proposed targets in the current evidence. The team should select a final candidate by explicitly weighing development-validation accuracy, operational cost and engineering priorities, not by declaring a single unconditional winner.

### Limitations and provenance
The development-validation accuracy figures in this notebook were loaded from the team's existing results/development/comparison.csv, generated with a different runtime (Python 3.12.9, scikit-learn 1.9.1). The computational figures were measured separately on my Python 3.10.6 / scikit-learn 1.7.2 Windows environment. The protocol and chosen configurations match, but cross-version differences mean these figures should be identified as combined evidence, not measurements from a single software environment. Wall-clock times vary with processor, system load, operating system and library implementations; they are not universal benchmarks. Memory increments based on sparse RSS samples are approximate and may understate actual peak memory. A zero recorded increment does not imply no memory use. No held-out 2014–2015 scores have been calculated.

### Actual contribution and AI disclosure
As Member 5, I configured and verified my own Python environment, checked the four models for basic compatibility, independently executed the four-model computational benchmark on my laptop, verified its saved runtime/protocol information and compared my timings with the team's recorded development accuracy. I interpreted the accuracy–latency trade-off and checked the provisional constraints. ChatGPT assisted with setup instructions, notebook verification/comparison code and drafting guidance. The common benchmark_all() implementation and older Linux reference run were generated during earlier Codex integration; I do not claim to have independently written that implementation or produced those earlier measurements. Final model selection and held-out testing require group agreement.